# Merge settlements_clean.csv and saws_clean.csv

In [ ]:
import pandas as pd
import numpy as np

settlements = pd.read_csv("settlements_clean.csv", low_memory = False)
saws = pd.read_csv("saws_clean.csv", parse_dates = ["date"])

print(f"Settlements: {settlements.shape}")
print(f"SAWS: {saws.shape}")

In [ ]:
print(f"Settlements columns: {settlements.columns.tolist()}")
print(f"SAWS columns: {saws.columns.tolist()}")

In [ ]:
saws_agg = saws.groupby(["station_id", "station_name", "lat", "lon", "elevation_m"]).agg(
    avg_max_temp_c = ("max_temp_c", "mean"),
    avg_min_temp_c = ("min_temp_c", "mean"),
    avg_temp_range_c = ("temp_range_c", "mean"),
    avg_cloud_octas = ("cloud_octas", "mean"),
    max_max_temp_c = ("max_temp_c", "max"),
    min_min_temp_c = ("min_temp_c", "min")
).reset_index()

print(f"SAWS aggregated: {saws_agg.shape}")
print(saws_agg[["station_name", "avg_max_temp_c", "avg_cloud_octas"]].to_string(index = False))

# Nearest Neighbour join

Use Haversine formula to return distance in km between two lat/lon points. Find the nearest SAWS station for each settlement

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371 # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def find_nearest_station(village_lat, village_lon, saws_agg):
    distances = saws_agg.apply(
        lambda row: haversine(village_lat, village_lon, row["lat"], row["lon"]),
        axis = 1
    )
    
    nearest_idx = distances.idxmin()
    
    return pd.Series({
        "nearest_station_id": saws_agg.loc[nearest_idx, "station_id"],
        "nearest_station_name": saws_agg.loc[nearest_idx, "station_name"],
        "nearest_station_km": round(distances[nearest_idx], 2)
    })

print("Finding nearest station for settlement...")
nearest = settlements.apply(
    lambda row: find_nearest_station(row["lat"], row["lon"], saws_agg),
    axis= 1
)
print("Done")
print(nearest.head())

Attach the nearest station results to the settlements and then merge in the aggreagted SAWS weather data

In [ ]:
settlements_with_station = pd.concat([settlements, nearest], axis = 1)

saws_agg_renamed = saws_agg.rename(columns = {
    "lat": "station_lat", 
    "lon": "station_lon",
})

settlements_saws = settlements_with_station.merge(
    saws_agg_renamed,
    left_on = "nearest_station_id",
    right_on = "station_id",
    how = "left"
)

# drop redundant station_id from saws_agg
settlements_saws = settlements_saws.drop(columns = ["station_id", "station_name"])

print(f"Merged shape: {settlements_saws.shape}")
print(f"\nNew weather columns added: {settlements_saws.columns.difference(settlements.columns).tolist()}")

In [ ]:
print(settlements_saws.head())

In [ ]:
print(settlements_saws[["village_name", "nearest_station_name", "nearest_station_km", "avg_max_temp_c", "avg_min_temp_c", "avg_temp_range_c", "avg_cloud_octas"]].head())

In [ ]:
print(settlements_saws.columns.tolist)

In [ ]:
settlements_saws.to_csv("settlements_saws.csv", index = False)
print(f"Shape: {settlements_saws.shape}")